Module 05 - Machine Learning Model Development

Objective

The objective of this module is to develop, train, and compare multiple machine learning models for credit risk prediction.

The following models are implemented:

- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- LightGBM

These models will later be compared using multiple evaluation metrics including ROC-AUC, Precision-Recall AUC, Gini Coefficient, F1 Score, Confusion Matrix and False Negative analysis.

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

import joblib

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/parth/CreditRiskAnalytics/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <4E82201A-ED82-3451-AD25-7886C77941A1> /Users/parth/CreditRiskAnalytics/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libom

X_train = pd.read_parquet(
    "../data/processed/X_train.parquet"
)

X_test = pd.read_parquet(
    "../data/processed/X_test.parquet"
)

y_train = pd.read_parquet(
    "../data/processed/y_train.parquet"
).squeeze()

y_test = pd.read_parquet(
    "../data/processed/y_test.parquet"
).squeeze()

In [ ]:
print("="*60)

print("Training Features :", X_train.shape)

print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)

print("Testing Labels :", y_test.shape)

print("="*60)

In [ ]:
print("Training Distribution")

print(
    y_train.value_counts(normalize=True)
)

print()

print("Testing Distribution")

print(
    y_test.value_counts(normalize=True)
)

Baseline Model 1 - Logistic Regression

Logistic Regression is trained as the baseline linear classification model. Class balancing is enabled to reduce the effect of dataset imbalance.

In [ ]:
logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

In [ ]:
%%time

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression Training Completed")

Baseline Model 2 - Decision Tree

A constrained Decision Tree is trained to provide an interpretable nonlinear baseline while minimizing overfitting.

In [ ]:
decision_tree = DecisionTreeClassifier(
    random_state=42,
    max_depth=12,
    min_samples_split=100,
    min_samples_leaf=50,
    class_weight="balanced"
)

In [ ]:
%%time

decision_tree.fit(
    X_train,
    y_train
)

print("Decision Tree Training Completed")

Baseline Model 3 - Random Forest

Random Forest is trained as an ensemble learning algorithm capable of capturing complex nonlinear relationships among borrower and loan characteristics.

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

In [ ]:
%%time

random_forest.fit(
    X_train,
    y_train
)

print("Random Forest Training Completed")

Advanced Model 1 - XGBoost

Extreme Gradient Boosting (XGBoost) is implemented as one of the primary gradient boosting algorithms recommended for structured tabular datasets.

In [ ]:
xgb_model = XGBClassifier(

    random_state=42,

    n_estimators=200,

    learning_rate=0.1,

    max_depth=6,

    subsample=0.8,

    colsample_bytree=0.8,

    eval_metric="logloss",

    n_jobs=-1
)

In [ ]:
%%time

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost Training Completed")

Advanced Model 2 - LightGBM

LightGBM is implemented as a gradient boosting framework optimized for large-scale structured datasets and is included as a mandatory model for comparison.

In [ ]:
lightgbm_model = LGBMClassifier(

    random_state=42,

    n_estimators=200,

    learning_rate=0.1,

    max_depth=6,

    class_weight="balanced"
)

In [ ]:
%%time

lightgbm_model.fit(
    X_train,
    y_train
)

print("LightGBM Training Completed")

Trained Models

All baseline and advanced machine learning models have now been successfully trained.

The evaluation phase will compare their predictive performance using multiple business and machine learning metrics.

In [ ]:
models = {

    "Logistic Regression": logistic_model,

    "Decision Tree": decision_tree,

    "Random Forest": random_forest,

    "XGBoost": xgb_model,

    "LightGBM": lightgbm_model

}

print("Models Trained Successfully")

print()

for model in models:

    print(model)

Model Evaluation

All trained machine learning models are evaluated using identical performance metrics to ensure a fair comparison.

The evaluation focuses not only on prediction accuracy but also on identifying the model that minimizes credit risk by reducing false negatives.

Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Precision-Recall AUC
- Gini Coefficient
- False Positive Rate
- False Negative Rate

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [ ]:
results = []

confusion_matrices = {}

classification_reports = {}

In [ ]:
for name, model in models.items():

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )

    pr_auc = average_precision_score(
        y_test,
        probabilities
    )

    gini = (2 * roc_auc) - 1

    false_negative_rate = fn / (fn + tp)

    false_positive_rate = fp / (fp + tn)

    results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "ROC AUC": roc_auc,

        "PR AUC": pr_auc,

        "Gini": gini,

        "False Positives": fp,

        "False Negatives": fn,

        "False Positive Rate": false_positive_rate,

        "False Negative Rate": false_negative_rate

    })

    confusion_matrices[name] = confusion_matrix(
        y_test,
        predictions
    )

    classification_reports[name] = classification_report(
        y_test,
        predictions,
        output_dict=True
    )

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

results_df.reset_index(
    drop=True,
    inplace=True
)

results_df

In [ ]:
results_df.style.format({

    "Accuracy":"{:.4f}",

    "Precision":"{:.4f}",

    "Recall":"{:.4f}",

    "F1 Score":"{:.4f}",

    "ROC AUC":"{:.4f}",

    "PR AUC":"{:.4f}",

    "Gini":"{:.4f}",

    "False Positive Rate":"{:.4f}",

    "False Negative Rate":"{:.4f}"

})

Confusion Matrices

The confusion matrix is analyzed for each model.

For credit risk prediction, minimizing False Negatives is particularly important because they represent risky borrowers incorrectly classified as safe borrowers.

In [ ]:
for name, matrix in confusion_matrices.items():

    print("="*60)

    print(name)

    print("="*60)

    print(matrix)

    print()

In [ ]:
for name, report in classification_reports.items():

    print("="*60)

    print(name)

    print("="*60)

    print(pd.DataFrame(report).transpose())

    print()

Business Interpretation

The evaluation focuses on identifying the model that provides the best balance between identifying defaulting borrowers and minimizing false alarms.

Special emphasis is placed on ROC-AUC, Precision-Recall AUC, Gini Coefficient and False Negative Rate since these metrics are more meaningful for credit risk assessment than overall accuracy alone.

In [ ]:
best_model = results_df.iloc[0]["Model"]

print("Best Baseline Model")

print(best_model)

In [ ]:
results_df.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")

In [ ]:
results_df

Baseline Model Comparison Summary

All baseline and advanced machine learning models have been evaluated.

The best-performing model identified through ROC-AUC will be selected for hyperparameter tuning in the next stage.